# graynet 2.0 output demo

This notebook demonstrates the current graynet output model:

- run a single-edge extraction
- inspect `run_metadata.json`
- read `edges_raw.parquet` and `roi_stats.parquet`
- subset one subject / feature / weight
- reconstruct a NetworkX graph


In [ ]:
from pathlib import Path

from graynet import (
    extract,
    roiwise_stats_indiv,
    load_run,
    get_edge_values,
    export_to_nx,
)


In [ ]:
repo_dir = Path.cwd().resolve().parent if Path.cwd().name == 'scripts' else Path.cwd().resolve()
example_dir = repo_dir / 'example_data' / 'freesurfer'
out_dir = repo_dir / 'example_data' / 'freesurfer' / 'demo_v2_outputs'
subjects = ['subject12345']

out_dir.mkdir(exist_ok=True, parents=True)
example_dir, out_dir


## Run graynet

The API returns the run directory path when `return_results=False`.


In [ ]:
edge_run_dir = extract(
    subjects,
    example_dir,
    base_feature='freesurfer_thickness',
    weight_method_list=['manhattan'],
    atlas='fsaverage',
    smoothing_param=10,
    out_dir=out_dir,
    return_results=False,
)

roi_run_dir = roiwise_stats_indiv(
    subjects,
    example_dir,
    base_feature='freesurfer_thickness',
    chosen_roi_stats=['median', 'mean'],
    atlas='fsaverage',
    smoothing_param=10,
    out_dir=out_dir,
    return_results=False,
)

edge_run_dir, roi_run_dir


## Inspect metadata


In [ ]:
edge_data, metadata = load_run(edge_run_dir)
metadata


## Read the canonical Parquet outputs


In [ ]:
edge_data.to_pandas().head()


In [ ]:
roi_run = load_run(roi_run_dir)
roi_run.roi_stats.to_pandas().head()


## Subset one subject / feature / weight


In [ ]:
subset = get_edge_values(
    edge_data,
    subject_id='subject12345',
    base_feature='freesurfer_thickness',
    weight_method='manhattan',
)
subset.to_pandas().head()


## Reconstruct a graph


In [ ]:
graph = export_to_nx(subset)
graph.number_of_nodes(), graph.number_of_edges()


## Next steps

From here you can:

- iterate subject-wise with `edge_data.iter_subjects()`
- compute graph measures from the reconstructed NetworkX graph
- pivot the edge table into matrices for machine learning
- export GraphML or CSV with `graynet export` if an older downstream tool expects them
